# Week 06 — Neural Network Surrogates

Retrain NN surrogates with the new W5 data points (each function now has +1 sample).

**Context for W6 (per W5 results):**
- 4 new bests (F5, F6, F7, F8) and 4 regressions (F1, F2, F3, F4).
- F5 keeps climbing reliably (now at 2307); models systematically underpredict.
- F2 has had 2 consecutive regressions; W4 outlier point at (0.90, 0.12) may be poisoning the MLP fit.
- F4 plateaued/regressed even with smaller step — peak is very narrow.
- F1, F3 still in the no-model-beats-baseline regime.

For each function:
1. 5-fold CV across `H ∈ {16, 32}` × variants `{plain, dropout, wd, ensemble}`
2. Pick best (H, variant) by CV RMSE
3. Compare against baseline `Y.std()` — only mark as 'usable model' if it beats baseline
4. Refit on all data, save to `../models/week_06/function_N_nn.pt`
5. Save per-dim gradient at current best point

**F1 extra:** retrain sign classifier with the new W5 sample.


In [1]:
import sys, json
from pathlib import Path
import numpy as np
sys.path.append('../src')
import nn_models as nm

MODELS_DIR = Path('../models/week_06')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

WIDTHS = [16, 32]
VARIANTS = list(nm.VARIANTS)

def load(n):
    X = np.load(f'../data/function_{n}/initial_inputs.npy')
    Y = np.load(f'../data/function_{n}/initial_outputs.npy')
    return X, Y

In [2]:
def search_and_save(n, verbose=True):
    X, Y = load(n)
    baseline = float(Y.std())
    results = []
    for H in WIDTHS:
        for v in VARIANTS:
            rmse = nm.cv_rmse(X, Y, v, hidden=H, n_splits=5, seed=0)
            results.append((rmse, H, v))
    results.sort(key=lambda r: r[0])
    best_rmse, best_H, best_v = results[0]
    improvement = (baseline - best_rmse) / baseline * 100
    beat_baseline = best_rmse < baseline

    if verbose:
        print(f'F{n}: {X.shape[0]} pts, {X.shape[1]}D, baseline RMSE = {baseline:.4f}')
        print(f'  All configs (sorted):')
        for r, H, v in results:
            mark = ' ← BEST' if (r, H, v) == (best_rmse, best_H, best_v) else ''
            print(f'    H={H:3d}  {v:10s}  CV RMSE = {r:.4f}{mark}')
        status = '✓ beats baseline' if beat_baseline else '✗ WORSE than baseline'
        print(f'  → best: H={best_H}, {best_v}, RMSE={best_rmse:.4f} ({improvement:+.1f}% vs baseline) {status}')

    # Fit final model on all data
    models, meta = nm.fit_final(X, Y, best_v, best_H)
    meta['cv_rmse'] = best_rmse
    meta['baseline_rmse'] = baseline
    meta['beats_baseline'] = beat_baseline
    meta['all_configs'] = [{'hidden': H, 'variant': v, 'rmse': r} for r, H, v in results]

    # Gradient at current best point
    x_best = X[Y.argmax()]
    grad = nm.gradient_at(models, meta, x_best)
    meta['gradient_at_best'] = grad.tolist()
    meta['x_best'] = x_best.tolist()
    meta['y_best'] = float(Y.max())

    if verbose:
        print(f'  Gradient dY/dx at best point: {np.round(grad, 3).tolist()}')

    nm.save(models, meta, MODELS_DIR / f'function_{n}_nn.pt')
    return meta

## Train all 8 functions

In [3]:
all_meta = {}
for n in range(1, 9):
    print('=' * 72)
    all_meta[n] = search_and_save(n, verbose=True)
    print()

F1: 15 pts, 2D, baseline RMSE = 0.0018
  All configs (sorted):
    H= 32  ensemble    CV RMSE = 0.0023 ← BEST
    H= 16  wd          CV RMSE = 0.0025
    H= 16  plain       CV RMSE = 0.0026
    H= 16  dropout     CV RMSE = 0.0029
    H= 16  ensemble    CV RMSE = 0.0029
    H= 32  dropout     CV RMSE = 0.0030
    H= 32  plain       CV RMSE = 0.0032
    H= 32  wd          CV RMSE = 0.0033
  → best: H=32, ensemble, RMSE=0.0023 (-27.6% vs baseline) ✗ WORSE than baseline


  Gradient dY/dx at best point: [0.05400000140070915, -0.035999998450279236]



F2: 15 pts, 2D, baseline RMSE = 0.2293
  All configs (sorted):
    H= 16  dropout     CV RMSE = 0.2089 ← BEST
    H= 32  dropout     CV RMSE = 0.2199
    H= 16  ensemble    CV RMSE = 0.2466
    H= 32  wd          CV RMSE = 0.2801
    H= 32  plain       CV RMSE = 0.2820
    H= 16  wd          CV RMSE = 0.2849
    H= 32  ensemble    CV RMSE = 0.2928
    H= 16  plain       CV RMSE = 0.3012
  → best: H=16, dropout, RMSE=0.2089 (+8.9% vs baseline) ✓ beats baseline
  Gradient dY/dx at best point: [0.6890000104904175, 0.41200000047683716]



F3: 20 pts, 3D, baseline RMSE = 0.0763
  All configs (sorted):
    H= 32  ensemble    CV RMSE = 0.0711 ← BEST
    H= 16  ensemble    CV RMSE = 0.0739
    H= 16  plain       CV RMSE = 0.0807
    H= 32  plain       CV RMSE = 0.0816
    H= 32  wd          CV RMSE = 0.0830
    H= 16  wd          CV RMSE = 0.0839
    H= 32  dropout     CV RMSE = 0.0940
    H= 16  dropout     CV RMSE = 0.0960
  → best: H=32, ensemble, RMSE=0.0711 (+6.9% vs baseline) ✓ beats baseline


  Gradient dY/dx at best point: [-0.052000001072883606, 0.09099999815225601, 0.527999997138977]



...

  Gradient dY/dx at best point: [-0.37700000405311584, -0.2290000021457672, -1.3220000267028809, -0.6240000128746033, -1.1979999542236328]



F7: 35 pts, 6D, baseline RMSE = 0.4865
  All configs (sorted):
    H= 32  ensemble    CV RMSE = 0.2932 ← BEST
    H= 32  dropout     CV RMSE = 0.3247
    H= 32  wd          CV RMSE = 0.3394
    H= 32  plain       CV RMSE = 0.3399
    H= 16  dropout     CV RMSE = 0.3517
    H= 16  plain       CV RMSE = 0.3868
    H= 16  wd          CV RMSE = 0.3883
    H= 16  ensemble    CV RMSE = 0.3886
  → best: H=32, ensemble, RMSE=0.2932 (+39.7% vs baseline) ✓ beats baseline


  Gradient dY/dx at best point: [-1.878000020980835, 0.14399999380111694, -1.3799999952316284, 0.871999979019165, -1.2410000562667847, 0.024000000208616257]



F8: 45 pts, 8D, baseline RMSE = 1.0967
  All configs (sorted):
    H= 32  plain       CV RMSE = 0.3880 ← BEST
    H= 16  ensemble    CV RMSE = 0.3939
    H= 32  wd          CV RMSE = 0.3941
    H= 32  ensemble    CV RMSE = 0.3977
    H= 16  dropout     CV RMSE = 0.4133
    H= 32  dropout     CV RMSE = 0.4465
    H= 16  wd          CV RMSE = 0.4974
    H= 16  plain       CV RMSE = 0.5108
  → best: H=32, plain, RMSE=0.3880 (+64.6% vs baseline) ✓ beats baseline


  Gradient dY/dx at best point: [0.48100000619888306, -0.2070000022649765, 0.6620000004768372, -0.16599999368190765, 0.9649999737739563, 0.4490000009536743, -0.23600000143051147, -0.28600001335144043]



## F1 sign classifier

F1 is numerically ~0 for almost all points. Training an NN classifier on sign(y > 0) gives the analyze step a map of "where is the function positive" — useful for Q3/Q6 in the reflection.

In [4]:
X1, Y1 = load(1)
pos_frac = (Y1 > 0).mean()
print(f'F1: {len(Y1)} pts, {(Y1 > 0).sum()} positive, {(Y1 <= 0).sum()} zero-or-negative ({pos_frac:.0%} positive)')

if 0 < pos_frac < 1:
    clf, loo_acc = nm.train_sign_classifier(X1, Y1, hidden=16)
    nm.save_classifier(clf, loo_acc, d_in=X1.shape[1], hidden=16, path=MODELS_DIR / 'function_1_classifier.pt')
    print(f'Sign classifier trained. LOO accuracy = {loo_acc:.2%}')
else:
    print('Classifier skipped (all samples are one class).')

F1: 15 pts, 11 positive, 4 zero-or-negative (73% positive)


Sign classifier trained. LOO accuracy = 73.33%


## Summary

In [5]:
print(f"{'F':>2}  {'H':>3}  {'variant':10s}  {'CV RMSE':>10s}  {'baseline':>10s}  {'improve%':>8s}  {'beats?':>6s}")
print('-' * 62)
for n, m in all_meta.items():
    improve = (m['baseline_rmse'] - m['cv_rmse']) / m['baseline_rmse'] * 100
    mark = '✓' if m['beats_baseline'] else '✗'
    print(f'{n:>2}  {m["hidden"]:>3}  {m["variant"]:10s}  {m["cv_rmse"]:>10.4f}  {m["baseline_rmse"]:>10.4f}  {improve:>+7.1f}%  {mark:>6s}')

 F    H  variant        CV RMSE    baseline  improve%  beats?
--------------------------------------------------------------
 1   32  ensemble        0.0023      0.0018    -27.6%       ✗
 2   16  dropout         0.2089      0.2293     +8.9%       ✓
 3   32  ensemble        0.0711      0.0763     +6.9%       ✓
 4   16  ensemble        4.3935      8.8649    +50.4%       ✓
 5   32  plain         149.6490    649.8841    +77.0%       ✓
 6   16  ensemble        0.3420      0.6021    +43.2%       ✓
 7   32  ensemble        0.2932      0.4865    +39.7%       ✓
 8   32  plain           0.3880      1.0967    +64.6%       ✓
